In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/akashbabuji/gesturerobottrainingdatapoints/cnn_training_dataset.csv


In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    "/kaggle/input/datasets/akashbabuji/gesturerobottrainingdatapoints/cnn_training_dataset.csv"
)

print(df.head())
print(df.shape)

         x0        y0            z0        x1        y1            z1  \
0  0.751541  0.985099  4.079779e-07  0.740608  0.937421  6.489063e-07   
1  0.740608  0.937421  6.489063e-07  0.719318  0.885217  5.961965e-07   
2  0.719318  0.885217  5.961965e-07  0.685744  0.813458  5.915793e-07   
3  0.685744  0.813458  5.915793e-07  0.649933  0.734880  6.163221e-07   
4  0.649933  0.734880  6.163221e-07  0.627213  0.666858  5.560903e-07   

         x2        y2            z2        x3  ...            z7        x8  \
0  0.719318  0.885217  5.961965e-07  0.685744  ...  5.228055e-07  0.574073   
1  0.685744  0.813458  5.915793e-07  0.649933  ...  4.783698e-07  0.567067   
2  0.649933  0.734880  6.163221e-07  0.627213  ...  4.426480e-07  0.563861   
3  0.627213  0.666858  5.560903e-07  0.606387  ...  4.016395e-07  0.561070   
4  0.606387  0.608391  5.801049e-07  0.584268  ...  3.928017e-07  0.559947   

         y8            z8        x9        y9            z9  target_x  \
0  0.517703  4.7836

In [3]:
# ============================================================
# CRMSF
# CNN REGRESSIVE MOTION STABILIZATION FILTER
# 10 EPOCH TRAINING
# ============================================================

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv1D,
    BatchNormalization,
    Dense,
    Dropout,
    Flatten
)
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint
)

# ============================================================
# LOAD DATASET
# ============================================================

# Replace path if needed

df = pd.read_csv(
     "/kaggle/input/datasets/akashbabuji/gesturerobottrainingdatapoints/cnn_training_dataset.csv"
)

print("Dataset Shape:", df.shape)

# ============================================================
# FEATURES / TARGETS
# ============================================================

X = df.iloc[:, 0:30].values
Y = df.iloc[:, 30:33].values

# ============================================================
# NORMALIZATION
# ============================================================

x_scaler = StandardScaler()
y_scaler = StandardScaler()

X = x_scaler.fit_transform(X)
Y = y_scaler.fit_transform(Y)

# ============================================================
# RESHAPE
# ============================================================

X = X.reshape(-1, 10, 3)

print("Input Shape :", X.shape)
print("Target Shape:", Y.shape)

# ============================================================
# TRAIN TEST SPLIT
# ============================================================

X_train, X_test, Y_train, Y_test = train_test_split(
    X,
    Y,
    test_size=0.20,
    random_state=42,
    shuffle=True
)

# ============================================================
# CRMSF NETWORK
# ============================================================

model = Sequential()

model.add(
    Conv1D(
        filters=32,
        kernel_size=3,
        activation='relu',
        input_shape=(10,3)
    )
)

model.add(
    BatchNormalization()
)

model.add(
    Conv1D(
        filters=64,
        kernel_size=3,
        activation='relu'
    )
)

model.add(
    BatchNormalization()
)

model.add(
    Conv1D(
        filters=128,
        kernel_size=3,
        activation='relu'
    )
)

model.add(
    Flatten()
)

model.add(
    Dense(
        128,
        activation='relu'
    )
)

model.add(
    Dropout(0.30)
)

model.add(
    Dense(
        64,
        activation='relu'
    )
)

# Output:
# predicted x,y,z

model.add(
    Dense(
        3,
        activation='linear'
    )
)

# ============================================================
# COMPILE
# ============================================================

model.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

model.summary()

# ============================================================
# CALLBACKS
# ============================================================

checkpoint = ModelCheckpoint(
    "CRMSF_Model.keras",
    monitor="val_loss",
    save_best_only=True,
    verbose=1
)

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

# ============================================================
# TRAIN
# ============================================================

history = model.fit(
    X_train,
    Y_train,
    validation_data=(
        X_test,
        Y_test
    ),
    epochs=10,
    batch_size=64,
    callbacks=[
        checkpoint,
        early_stop
    ],
    verbose=1
)

# ============================================================
# EVALUATE
# ============================================================

loss, mae = model.evaluate(
    X_test,
    Y_test,
    verbose=0
)

print("\n==========================")
print("FINAL RESULTS")
print("==========================")
print("Test MSE :", loss)
print("Test MAE :", mae)

# ============================================================
# SAVE FINAL MODEL
# ============================================================

model.save(
    "CRMSF_Final.keras"
)

print("\nSaved:")
print("CRMSF_Model.keras")
print("CRMSF_Final.keras")

# ============================================================
# TEST ONE SAMPLE
# ============================================================

sample = X_test[0].reshape(1,10,3)

prediction = model.predict(
    sample,
    verbose=0
)

prediction = y_scaler.inverse_transform(
    prediction
)

actual = y_scaler.inverse_transform(
    Y_test[0].reshape(1,-1)
)

print("\nPrediction:", prediction[0])
print("Actual    :", actual[0])

2026-06-19 18:35:08.769474: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781894108.932916      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781894108.978245      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1781894109.353328      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781894109.353371      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781894109.353374      23 computation_placer.cc:177] computation placer alr

Dataset Shape: (237, 33)
Input Shape : (237, 10, 3)
Target Shape: (237, 3)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
I0000 00:00:1781894120.940934      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 8, 32)          │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 8, 32)          │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 6, 64)          │         6,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 6, 64)          │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 4, 128)         │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        65,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

...

Epoch 1/10


I0000 00:00:1781894124.478049      65 service.cc:152] XLA service 0x7da32c0055b0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1781894124.478102      65 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1781894124.982267      65 cuda_dnn.cc:529] Loaded cuDNN version 91002


1/3 ━━━━━━━━━━━━━━━━━━━━ 10s 5s/step - loss: 0.8581 - mae: 0.7554

I0000 00:00:1781894127.489182      65 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - loss: 0.8865 - mae: 0.7242 
Epoch 1: val_loss improved from None to 1.14678, saving model to CRMSF_Model.keras

Epoch 1: finished saving model to CRMSF_Model.keras
3/3 ━━━━━━━━━━━━━━━━━━━━ 8s 2s/step - loss: 0.8622 - mae: 0.6928 - val_loss: 1.1468 - val_mae: 0.7780
Epoch 2/10
1/3 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - loss: 0.4098 - mae: 0.4921
Epoch 2: val_loss improved from 1.14678 to 1.14009, saving model to CRMSF_Model.keras

Epoch 2: finished saving model to CRMSF_Model.keras
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step - loss: 0.5266 - mae: 0.5244 - val_loss: 1.1401 - val_mae: 0.7782
Epoch 3/10
1/3 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - loss: 0.4682 - mae: 0.4844
Epoch 3: val_loss improved from 1.14009 to 1.12711, saving model to CRMSF_Model.keras

Epoch 3: finished saving model to CRMSF_Model.keras
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step - loss: 0.4866 - mae: 0.4910 - val_loss: 1.1271 - val_mae: 0.7723
Epoch 4/10
1/3 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - los

In [4]:
# ============================================================
# ADVANCED CRMSF
# CNN REGRESSIVE MOTION STABILIZATION FILTER
# PUBLICATION VERSION
# 100 EPOCH TRAINING
# ============================================================

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import tensorflow as tf

from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input,
    Conv1D,
    BatchNormalization,
    Activation,
    Add,
    GlobalAveragePooling1D,
    Dense,
    Dropout,
    Concatenate,
    MultiHeadAttention,
    LayerNormalization
)

from tensorflow.keras.callbacks import (
    ModelCheckpoint,
    EarlyStopping,
    ReduceLROnPlateau
)

# ============================================================
# LOAD DATASET
# ============================================================

df = pd.read_csv(
    "/kaggle/input/datasets/akashbabuji/gesturerobottrainingdatapoints/cnn_training_dataset.csv"
)

print("Dataset:", df.shape)

# ============================================================
# FEATURES
# ============================================================

X = df.iloc[:,0:30].values
Y = df.iloc[:,30:33].values

# ============================================================
# NORMALIZATION
# ============================================================

x_scaler = StandardScaler()
y_scaler = StandardScaler()

X = x_scaler.fit_transform(X)
Y = y_scaler.fit_transform(Y)

X = X.reshape(-1,10,3)

# ============================================================
# SPLIT
# ============================================================

X_train,X_test,Y_train,Y_test = train_test_split(
    X,
    Y,
    test_size=0.20,
    random_state=42,
    shuffle=True
)

print(X_train.shape)
print(Y_train.shape)

# ============================================================
# RESIDUAL BLOCK
# ============================================================

def residual_block(x, filters):

    shortcut = x

    x = Conv1D(
        filters,
        3,
        padding='same'
    )(x)

    x = BatchNormalization()(x)

    x = Activation('relu')(x)

    x = Conv1D(
        filters,
        3,
        padding='same'
    )(x)

    x = BatchNormalization()(x)

    if shortcut.shape[-1] != filters:

        shortcut = Conv1D(
            filters,
            1,
            padding='same'
        )(shortcut)

    x = Add()([x, shortcut])

    x = Activation('relu')(x)

    return x

# ============================================================
# MODEL
# ============================================================

inputs = Input(shape=(10,3))

# Multi-scale branch

b1 = Conv1D(
    32,
    3,
    padding='same',
    activation='relu'
)(inputs)

b2 = Conv1D(
    32,
    5,
    padding='same',
    activation='relu'
)(inputs)

b3 = Conv1D(
    32,
    7,
    padding='same',
    activation='relu'
)(inputs)

x = Concatenate()([b1,b2,b3])

# Residual feature extraction

x = residual_block(x,64)
x = residual_block(x,128)
x = residual_block(x,128)

# Self attention

attn = MultiHeadAttention(
    num_heads=4,
    key_dim=32
)(x,x)

x = Add()([x,attn])

x = LayerNormalization()(x)

# Pooling

x = GlobalAveragePooling1D()(x)

# Dense regression

x = Dense(
    256,
    activation='relu'
)(x)

x = Dropout(0.30)(x)

x = Dense(
    128,
    activation='relu'
)(x)

x = Dropout(0.20)(x)

x = Dense(
    64,
    activation='relu'
)(x)

outputs = Dense(
    3,
    activation='linear'
)(x)

model = Model(
    inputs,
    outputs
)

# ============================================================
# COMPILE
# ============================================================

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-3
    ),
    loss='mse',
    metrics=['mae']
)

model.summary()

# ============================================================
# CALLBACKS
# ============================================================

checkpoint = ModelCheckpoint(
    "CRMSF_BEST.keras",
    monitor='val_loss',
    save_best_only=True,
    verbose=1
)

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    verbose=1
)

# ============================================================
# TRAIN
# ============================================================

history = model.fit(
    X_train,
    Y_train,
    validation_data=(
        X_test,
        Y_test
    ),
    epochs=100,
    batch_size=128,
    callbacks=[
        checkpoint,
        early_stop,
        reduce_lr
    ],
    verbose=1
)

# ============================================================
# EVALUATE
# ============================================================

loss, mae = model.evaluate(
    X_test,
    Y_test,
    verbose=0
)

print("\n==========================")
print("ADVANCED CRMSF RESULTS")
print("==========================")
print("Test MSE :", loss)
print("Test MAE :", mae)

# ============================================================
# SAVE
# ============================================================

model.save(
    "CRMSF_Publication_Model.keras"
)

print("\nSaved:")
print("CRMSF_BEST.keras")
print("CRMSF_Publication_Model.keras")

Dataset: (237, 33)
(189, 10, 3)
(189, 3)


Model: "functional_10"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 10, 3)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_3 (Conv1D)   │ (None, 10, 32)    │        320 │ input_layer_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_4 (Conv1D)   │ (None, 10, 32)    │        512 │ input_layer_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_5 (Conv1D)   │ (None, 10, 32)    │        704 │ input_layer_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 10, 96)    │          0 │ conv1d_3[0][0],   │
│ (Concatenate)       │                   │            │ conv1d_4[0][0],   │
│                     │                   │            │ conv1d_5[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_6 (Conv1D)   │ (None, 10, 64)    │     18,496 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 10, 64)    │        256 │ conv1d_6[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 10, 64)    │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_7 (Conv1D)   │ (None, 10, 64)    │     12,352 │ activation[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 10, 64)    │        256 │ conv1d_7[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_8 (Conv1D)   │ (None, 10, 64)    │      6,208 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 10, 64)    │          0 │ batch_normalizat… │
│                     │                   │            │ conv1d_8[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, 10, 64)    │          0 │ add[0][0]         │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_9 (Conv1D)   │ (None, 10, 128)   │     24,704 │ activation_1[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 10, 128)   │        512 │ conv1d_9[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_2        │ (None, 10, 128)   │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_10 (Conv1D)  │ (None, 10, 128)   │     49,280 │ activation_2[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 10, 128)   │        512 │ conv1d_10[0][0]   │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_11 (Conv1D)  │ (None, 10, 128)   │      8,320 │ activation_1[0][… │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 362,691 (1.38 MB)

 Trainable params: 361,411 (1.38 MB)

 Non-trainable params: 1,280 (5.00 KB)

Epoch 1/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 6s/step - loss: 1.2397 - mae: 0.9207  
Epoch 1: val_loss improved from None to 1.07508, saving model to CRMSF_BEST.keras

Epoch 1: finished saving model to CRMSF_BEST.keras
2/2 ━━━━━━━━━━━━━━━━━━━━ 21s 7s/step - loss: 1.1204 - mae: 0.8646 - val_loss: 1.0751 - val_mae: 0.7475 - learning_rate: 0.0010
Epoch 2/100
1/2 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - loss: 0.7260 - mae: 0.6396
Epoch 2: val_loss improved from 1.07508 to 0.98035, saving model to CRMSF_BEST.keras

Epoch 2: finished saving model to CRMSF_BEST.keras
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 280ms/step - loss: 0.6748 - mae: 0.6107 - val_loss: 0.9804 - val_mae: 0.6935 - learning_rate: 0.0010
Epoch 3/100
1/2 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - loss: 0.5682 - mae: 0.5284
Epoch 3: val_loss improved from 0.98035 to 0.90067, saving model to CRMSF_BEST.keras

Epoch 3: finished saving model to CRMSF_BEST.keras
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 260ms/step - loss: 0.5106 - mae: 0.5060 - val_loss: 0.9007 - val_mae: 0